**Dimensionality Reduction and Feature Selection**

Part 1: PCA and Variance Threshold in a Linear Regression

Import the housing data as a data frame and ensure that the data is loaded properly.

In [56]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold

In [48]:
import pandas as pd

df = pd.read_csv("./data/housing.csv", encoding = "utf-8")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Drop the "Id" column and any features that are missing more than 40% of their values.

In [49]:
df = df.drop(columns = ["Id"])

In [50]:
features = list(df.columns)
for feature in features:
    x = f"{feature}"
    null = df[df[x].isnull()]
    percent = len(null)/len(df)
    if percent > 0.4:
        print(x)

Alley
MasVnrType
FireplaceQu
PoolQC
Fence
MiscFeature


In [51]:
df = df.drop(columns = ["Alley", "MasVnrType", "FireplaceQu", "PoolQC", "Fence", "MiscFeature"])
df.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,Gtl,...,0,0,0,0,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,Gtl,...,0,0,0,0,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,Gtl,...,0,0,0,0,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,Gtl,...,272,0,0,0,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,Gtl,...,0,0,0,0,0,12,2008,WD,Normal,250000


For numerical columns, fill in any missing data with the median value.

In [52]:
features = list(df.columns)
import numpy as np
for feature in features:
    x = f"{feature}"
    dtype = df[x].dtype
    if pd.api.types.is_numeric_dtype(df[x]):       
        df[x] = df[x].fillna(df[x].median())

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 74 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   str    
 2   LotFrontage    1460 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   str    
 5   LotShape       1460 non-null   str    
 6   LandContour    1460 non-null   str    
 7   Utilities      1460 non-null   str    
 8   LotConfig      1460 non-null   str    
 9   LandSlope      1460 non-null   str    
 10  Neighborhood   1460 non-null   str    
 11  Condition1     1460 non-null   str    
 12  Condition2     1460 non-null   str    
 13  BldgType       1460 non-null   str    
 14  HouseStyle     1460 non-null   str    
 15  OverallQual    1460 non-null   int64  
 16  OverallCond    1460 non-null   int64  
 17  YearBuilt      1460 non-null   int64  
 18  YearRemodAdd   1460

For categorical columns, fill in any missing data with the most common value (mode).

In [53]:
features = list(df.columns)
for feature in features:
    x = f"{feature}"
    dtype = df[x].dtype
    if pd.api.types.is_string_dtype(df[x]):       
        df[x] = df[x].fillna(df[x].mode()[0])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 74 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   str    
 2   LotFrontage    1460 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   str    
 5   LotShape       1460 non-null   str    
 6   LandContour    1460 non-null   str    
 7   Utilities      1460 non-null   str    
 8   LotConfig      1460 non-null   str    
 9   LandSlope      1460 non-null   str    
 10  Neighborhood   1460 non-null   str    
 11  Condition1     1460 non-null   str    
 12  Condition2     1460 non-null   str    
 13  BldgType       1460 non-null   str    
 14  HouseStyle     1460 non-null   str    
 15  OverallQual    1460 non-null   int64  
 16  OverallCond    1460 non-null   int64  
 17  YearBuilt      1460 non-null   int64  
 18  YearRemodAdd   1460

Convert the categorical columns to dummy variables.

In [54]:
df = pd.get_dummies(df, drop_first = True)

Split the data into a training and test set, where the SalePrice column is the target.

In [57]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="SalePrice")
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Run a linear regression and report the R2-value and RMSE on the test set.

In [58]:
model = LinearRegression()

model.fit(X_train, y_train)

pred = model.predict(X_test)

r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("R²:", r2)
print("RMSE:", rmse)

R²: 0.8786592275925837
RMSE: 30507.758137839846


Fit and transform the training features with a PCA so that 90% of the variance is retained (see section 9.1 in the Machine Learning with Python Cookbook).

In [63]:
pca = PCA(n_components=0.90)
X_train_pca = pca.fit_transform(X_train)

How many features are in the PCA-transformed matrix?

In [64]:
print("Number of PCA features:", X_train_pca.shape[1])

Number of PCA features: 1


Transform but DO NOT fit the test features with the same PCA.

In [65]:
X_test_pca = pca.transform(X_test)

Repeat step 7 with your PCA transformed data.

Take your original training features (from step 6) and apply a min-max scaler to them.

Find the min-max scaled features in your training set that have a variance above 0.1 (see Section 10.1 in the Machine Learning with Python Cookbook).

Transform but DO NOT fit the test features with the same steps applied in steps 11 and 12.

Repeat step 7 with the high variance data.

Summarize your findings.

Part 2: Categorical Feature Selection



Download the data from this link Mushroom Classification. Based on several categorical features, you will predict whether or not a mushroom is edible or poisonous.

Import the data as a data frame and ensure it is loaded correctly.
Convert the categorical features (all of them) to dummy variables.
Split the data into a training and test set.
Fit a decision tree classifier on the training set.
Report the accuracy and create a confusion matrix for the model prediction on the test set.
Create a visualization of the decision tree.
Use a χ2-statistic selector to pick the five best features for this data (see section 10.4 of the Machine Learning with Python Cookbook).
Which five features were selected in step 7? Hint: Use the get_support function.
Repeat steps 4 and 5 with the five best features selected in step 7.
Summarize your findings.